# Transformer-Based Classification of Contract Clause Themes

**Capstone Project 4 — Deep Learning Systems**

This notebook implements a PyTorch Transformer experiment using the Contract Understanding Atticus Dataset (CUAD). The project adapts expert-annotated contract passages into a ten-class text-classification task that demonstrates a foundation for ICM Solutions knowledge reuse and document-quality workflows.

## Phase 4 setup objective

This initial version validates the Google Colab environment, GPU availability, repository data access, and dataset integrity. Model implementation and training sections will be added after the environment is confirmed.

## 1. Environment and library validation

The notebook uses standard Python data-science and PyTorch libraries required by the project rubric.

In [ ]:
import os
import platform
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
    import sklearn
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
    SKLEARN_VERSION = sklearn.__version__
except ImportError as exc:
    raise ImportError("scikit-learn is required for evaluation metrics.") from exc

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"scikit-learn: {SKLEARN_VERSION}")

## 2. Reproducibility settings

A fixed seed is applied to Python, NumPy, and PyTorch. GPU operations can still show small numerical differences across hardware and runtime versions.

In [ ]:
SEED = 42

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Random seed set to {SEED}.")

## 3. Device validation

Google Colab should be configured to use a GPU runtime. CUDA is preferred, but the notebook retains a CPU fallback.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Selected device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version reported by PyTorch: {torch.version.cuda}")
    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU memory: {total_memory_gb:.2f} GB")
else:
    print("CUDA is not available. In Colab, select Runtime > Change runtime type > GPU.")

## 4. Load the processed CUAD classification dataset

The processed dataset is stored in the public GitHub repository. Loading it from the raw GitHub URL makes the notebook portable across Colab and local environments without requiring a hard-coded personal path.

In [ ]:
REPOSITORY = "icmsol/capstone-project-4-deep-learning-systems"
DATA_URL = (
    "https://raw.githubusercontent.com/"
    f"{REPOSITORY}/main/data/processed/cuad_clause_classification.csv"
)

required_columns = {
    "example_id",
    "contract_id",
    "category",
    "label_id",
    "clause_text",
    "word_count",
    "char_count",
    "fold",
    "split",
}

df = pd.read_csv(DATA_URL)

missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

print(f"Loaded dataset from: {DATA_URL}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
display(df.head(3))

## 5. Dataset integrity checks

The split is grouped by source contract. These checks verify that no contract appears in more than one of the training, validation, or test partitions.

In [ ]:
expected_splits = {"train", "validation", "test"}
observed_splits = set(df["split"].unique())

if observed_splits != expected_splits:
    raise ValueError(
        f"Expected splits {sorted(expected_splits)}, but found {sorted(observed_splits)}."
    )

split_summary = (
    df.groupby("split")
      .agg(
          passages=("example_id", "count"),
          contracts=("contract_id", "nunique"),
          categories=("category", "nunique"),
      )
      .reindex(["train", "validation", "test"])
)

train_contracts = set(df.loc[df["split"] == "train", "contract_id"])
validation_contracts = set(df.loc[df["split"] == "validation", "contract_id"])
test_contracts = set(df.loc[df["split"] == "test", "contract_id"])

overlap_checks = {
    "train_validation_overlap": len(train_contracts & validation_contracts),
    "train_test_overlap": len(train_contracts & test_contracts),
    "validation_test_overlap": len(validation_contracts & test_contracts),
}

if any(overlap_checks.values()):
    raise ValueError(f"Contract leakage detected: {overlap_checks}")

print("Split summary:")
display(split_summary)
print("Contract overlap checks:", overlap_checks)
print("Dataset integrity checks passed.")

## 6. Preliminary class and length inspection

These outputs confirm that all ten selected categories are present and provide an initial view of text-length variability.

In [ ]:
category_summary = (
    df.groupby(["label_id", "category"])
      .agg(
          passages=("example_id", "count"),
          contracts=("contract_id", "nunique"),
          median_words=("word_count", "median"),
          max_words=("word_count", "max"),
      )
      .reset_index()
      .sort_values("label_id")
)

display(category_summary)

fig, ax = plt.subplots(figsize=(10, 5))
category_counts = df["category"].value_counts().sort_values()
category_counts.plot(kind="barh", ax=ax)
ax.set_title("CUAD Classification Dataset — Passages per Category")
ax.set_xlabel("Number of passages")
ax.set_ylabel("Clause category")
plt.tight_layout()
plt.show()

## Phase 4 checkpoint

The environment setup is complete when:

1. All imports succeed.
2. CUDA is detected in the Colab runtime.
3. The processed dataset loads directly from GitHub.
4. The dataset contains 4,356 passages and ten categories.
5. All contract-overlap checks return zero.

The remaining notebook sections will implement preprocessing, the baseline Transformer, the controlled dropout experiment, evaluation, error analysis, and the required 4–6 sentence project summary.